<a href="https://colab.research.google.com/github/rt956gh-crypto/-/blob/main/SDDB_Week07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📋 規格書（請直接在這一格修改）

學號：411303091　姓名：古家原

## 1. 系統目的（一句話）
讓民眾用自然語言完成診所內科、外科、牙科的掛號、取消掛號與剩餘名額查詢。

## 2. 使用者會怎麼說（至少 5 句）
1.我想掛號內科
2.幫我取消牙科的掛號
3.請幫我查一下外科還有沒有名額
4.我牙痛想掛號
5.我今天肚子好痛，該掛什麼科？

## 3. 資料表（最多 2 張）
| 表名 | 欄位名稱 | 型態 | 說明 |
|---|---|---|---|
| departments | dept_name | TEXT | 科別名稱（主鍵，如：內科、外科、牙科） |
| departments | quota | INTEGER | 該科別當日剩餘名額 |
| appointments | id | INTEGER | 掛號紀錄編號（主鍵，自動遞增） |
| appointments | dept_name | TEXT | 掛號科別 |
| appointments | status | TEXT | 掛號狀態（如：已掛號、已取消） |

初始資料：
departments:

('內科', 5)

('外科', 3)

('牙科', 0)（預設額滿，用於測試邊界條件）

appointments: 初始為空表

## 4. AI 的工作
- 輸入：使用者的一句話
- 輸出格式：動作,對象（例：掛號,內科）
- 動作只能是：掛號、取消、查詢、未知
- 對象只能是：內科、外科、牙科、未知
- 無法判斷時輸出：`未知,未知`
- AI 不可以做的事：
1.嚴禁提供任何醫療建議、診斷或推測病因。
2.嚴禁根據使用者描述的症狀推薦或自動判定科別（即使使用者說「肚子痛」或「牙痛」，若未明確指出科別，一律輸出 未知,未知）。

## 5. 功能與規則
| 功能 | 條件 | 系統回應 | 資料庫變化 |
|---|---|---|---|
| 掛號 | 該科別名額 > 0 | 「掛號成功！已為您掛號［科別］。」 | 1. 新增一筆記錄至 `appointments`<br>2. `departments` 該科別 `quota` 減 1 |
| 掛號 | 該科別名額 = 0 | 「掛號失敗！［科別］今日名額已滿。」 | 無變化 |
| 取消 | 存在狀態為 `已掛號` 的該科紀錄 | 「取消成功！已為您取消［科別］掛號。」 | 1. 將該筆 `appointments` 狀態更新為 `已取消`<br>2. `departments` 該科別 `quota` 加 1 |
| 取消 | 無該科別之已掛號紀錄 | 「取消失敗！查無您的［科別］掛號紀錄。」 | 無變化 |
| 查詢 | 無條件限制 | 「［科別］今日剩餘名額為：［名額］人。」 | 無變化 |
| 未知處理 | AI 回傳包含 `未知` | 「抱歉，無法識別您的意圖或科別。本系統僅提供內科、外科、牙科之掛號服務，且不提供醫療建議。」 | 無變化 |

## 6. 例外情況（至少 3 個）
1.科別名額不足：使用者欲掛號的科別剩餘名額為 0 時，系統拒絕掛號並提示額滿。
2.無紀錄可取消：使用者要求取消掛號，但資料庫中找不到該科別之有效掛號紀錄時，系統提示取消失敗。
3.僅描述症狀未講科別：使用者詢問症狀（例如：「我頭痛要看哪科？」），系統拒絕給予診斷 advice，並回覆無法辨識意圖。

## 7. 測試清單（至少 8 句，正常／錯誤／邊界都要有）
| # | 類型 | 使用者說的話 | 預期結果 |
|---|---|---|---|
| 1 | 正常 | 我要掛號內科 | AI 回傳 `掛號,內科`；系統顯示掛號成功，內科名額減 1 |
| 2 | 正常 | 請幫我取消內科掛號 | AI 回傳 `取消,內科`；系統顯示取消成功，內科名額加 1 |
| 3 | 正常 | 查詢外科剩餘名額 | AI 回傳 `查詢,外科`；系統顯示外科剩餘名額 |
| 4 | 邊界 | 我要掛牙科 | AI 回傳 `掛號,牙科`；因牙科名額為 0，系統顯示「掛號失敗！牙科今日名額已滿」 |
| 5 | 錯誤 | 幫我取消眼科 | AI 回傳 `取消,未知`（眼科不在服務範圍）；系統顯示無法識別意圖 |
| 6 | 錯誤 | 我要取消牙科掛號 | AI 回傳 `取消,牙科`；因無牙科掛號紀錄，系統顯示「取消失敗！查無您的牙科掛號紀錄」 |
| 7 | 邊界 | 我牙痛想掛號 | AI 回傳 `未知,未知`；系統顯示無法識別意圖且不給予建議 |
| 8 | 錯誤 | 今天天氣真好 | AI 回傳 `未知,未知`；系統顯示無法識別意圖 |